# CayleyPy checkpoint-only beam search -- exactly 2xT4

This is a thin, reader-facing launcher for **standard CayleyPy competitions**:
the attached input must contain `puzzle_info.json`, `test.csv`, and
`sample_submission.csv`. It runs the repository's existing two-rank beam-search
implementation; this notebook does not contain a second solver.

## Fixed model contract

- Runtime: exactly **two Tesla/NVIDIA T4 GPUs**; the run fails before export if
  the hardware differs.
- Models: checkpoint-only, with automatic checkpoint-format detection. Supported
  families are **batchnorm-folded** and **resmlp-layernorm**.
- Model dtype is automatically **fp16**. The head must be exactly
  `output_dim=1` or `output_dim=move_count` for the selected puzzle type.
- The fixed public State128 runner requires `1 <= state_len <= 120` and the
  permutation contract requires `num_classes=state_len`, with every central and
  initial-state label in `0..state_len-1`.
- The inclusive `PUZZLE_ID_START..PUZZLE_ID_END` range is checked strictly:
  a missing ID is a configuration error.

## Search contract

Your requested beam is kept, except for documented two-rank/shard alignment.
The nearest measured 2xT4 profile in `2**16..2**25` is selected automatically.
You may choose reflection `off`, `after_original`, or `only`; choose either first
solution or collection through a chosen depth; and set the touch-BFS radius.
CUDA/C++ algorithm knobs are deliberately absent.

## Results publishing

Publishing is best effort: valid local solutions and `submission.csv` remain even
if the ingest service is unavailable. The status is saved in `publish_status.json`.
No token or secret belongs in this notebook.
Publishing remains explicit opt-in. Set an HTTPS endpoint in the config or leave
it blank to use `CAYLEYPY_RESULTS_INGEST_URL`; no endpoint is bundled. Publishing
fails closed before solve if author or competition/Kaggle provenance still uses
the supplied `replace-with-*` placeholders.
The endpoint must be a public HTTPS host: localhost and non-global IP literals are
rejected. Redirects are never followed, so a configured origin cannot redirect
the publisher to a private or plaintext destination.

### Provenance hash rule

`KAGGLE_NOTEBOOK_SHA256` is the SHA-256 of canonical immutable **cell sources**:
the header, preflight, CLI invocation, and artifact-display cells, in notebook
order. The user configuration and the provenance-computing setup cell are excluded
to avoid a self-referential hash. Each source is UTF-8, normalized to LF with one
trailing LF, and joined using one LF. The setup cell computes the value and the
CLI records it only when publishing is enabled.

In [ ]:
from pathlib import Path

# USER CONFIG -- edit only these values. Explicit Kaggle paths prevent ambiguous
# auto-discovery across several attached competitions/models.
CHECKPOINT_PATH = Path("/kaggle/input/models/arabidopsisthalian/megaminx2048-512-8-e4000/pytorch/default/1/weights_megaminx2048_512_8_e4000.pth")
PUZZLE_INFO_JSON = Path("/kaggle/input/competitions/cayley-py-megaminx/puzzle_info.json")
TEST_CSV = Path("/kaggle/input/competitions/cayley-py-megaminx/test.csv")
SAMPLE_SUBMISSION_CSV = Path("/kaggle/input/competitions/cayley-py-megaminx/sample_submission.csv")

PUZZLE_ID_START = 0                 # inclusive
PUZZLE_ID_END = 0                   # inclusive
BEAM_WIDTH = 65536                  # requested value; only alignment may increase it
MAX_DEPTH = 8
REFLECT_MODE = "off"                # off | after_original | only
REFLECT_SOURCE_CSV = None            # required only for REFLECT_MODE == "only"
SOLUTION_MODE = "first"             # first | collect
COLLECT_UNTIL_DEPTH = MAX_DEPTH      # meaningful when SOLUTION_MODE == "collect"
MAX_COLLECTED_SOLUTIONS = 100
TOUCH_BFS_RADIUS = 0

AUTHOR_NAME = "acceptance-test"
PUBLISH_RESULTS = False              # explicit opt-in; failures keep local artifacts
# Explicit HTTPS endpoint wins; blank falls back to CAYLEYPY_RESULTS_INGEST_URL.
RESULTS_INGEST_URL = ""              # no endpoint, token, or secret is bundled
COMPETITION = "cayley-py-megaminx"
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cayleypy-public-acceptance-smoke-output1"
KAGGLE_VERSION = 1
KAGGLE_USERNAME = "trydotatwo"

# Official repository revision. Keep this 40-character commit pinned for a
# reproducible solve; SOLVER_COMMIT always refers to TryDotAtwo/MultiGPUBeamSearch.
SOLVER_COMMIT = "d55af018a8f3336e3d0f22bb2d54144abbf262da"

In [ ]:
# Checkout only the pinned public solver revision and derive notebook provenance.
from hashlib import sha256
import json
from pathlib import Path
import shutil
import subprocess

WORK = Path("/kaggle/working/cayleypy_public")
REPO = WORK / "MultiGPUBeamSearch"
CONFIG_PATH = WORK / "run_config.json"
OUTPUT_DIR = WORK / "output"
OFFICIAL_SOLVER_REPOSITORY = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"

if SOLVER_COMMIT == "0" * 40:
    raise ValueError("set SOLVER_COMMIT to a real 40-character public commit")
for required_path in (CHECKPOINT_PATH, PUZZLE_INFO_JSON, TEST_CSV, SAMPLE_SUBMISSION_CSV):
    if "REPLACE_WITH_" in str(required_path):
        raise ValueError(f"replace the explicit Kaggle path: {required_path}")
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

WORK.mkdir(parents=True, exist_ok=True)
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", OFFICIAL_SOLVER_REPOSITORY, str(REPO)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOLVER_COMMIT], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
checked_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if checked_commit != SOLVER_COMMIT:
    raise RuntimeError(f"pinned commit mismatch: expected={SOLVER_COMMIT} got={checked_commit}")

# Canonical notebook-source hash: immutable cells only (not user config/setup).
NOTEBOOK_CELL_SOURCES = ["# CayleyPy checkpoint-only beam search -- exactly 2xT4\n\nThis is a thin, reader-facing launcher for **standard CayleyPy competitions**:\nthe attached input must contain `puzzle_info.json`, `test.csv`, and\n`sample_submission.csv`. It runs the repository's existing two-rank beam-search\nimplementation; this notebook does not contain a second solver.\n\n## Fixed model contract\n\n- Runtime: exactly **two Tesla/NVIDIA T4 GPUs**; the run fails before export if\n  the hardware differs.\n- Models: checkpoint-only, with automatic checkpoint-format detection. Supported\n  families are **batchnorm-folded** and **resmlp-layernorm**.\n- Model dtype is automatically **fp16**. The head must be exactly\n  `output_dim=1` or `output_dim=move_count` for the selected puzzle type.\n- The fixed public State128 runner requires `1 <= state_len <= 120` and the\n  permutation contract requires `num_classes=state_len`, with every central and\n  initial-state label in `0..state_len-1`.\n- The inclusive `PUZZLE_ID_START..PUZZLE_ID_END` range is checked strictly:\n  a missing ID is a configuration error.\n\n## Search contract\n\nYour requested beam is kept, except for documented two-rank/shard alignment.\nThe nearest measured 2xT4 profile in `2**16..2**25` is selected automatically.\nYou may choose reflection `off`, `after_original`, or `only`; choose either first\nsolution or collection through a chosen depth; and set the touch-BFS radius.\nCUDA/C++ algorithm knobs are deliberately absent.\n\n## Results publishing\n\nPublishing is best effort: valid local solutions and `submission.csv` remain even\nif the ingest service is unavailable. The status is saved in `publish_status.json`.\nNo token or secret belongs in this notebook.\nPublishing remains explicit opt-in. Set an HTTPS endpoint in the config or leave\nit blank to use `CAYLEYPY_RESULTS_INGEST_URL`; no endpoint is bundled. Publishing\nfails closed before solve if author or competition/Kaggle provenance still uses\nthe supplied `replace-with-*` placeholders.\nThe endpoint must be a public HTTPS host: localhost and non-global IP literals are\nrejected. Redirects are never followed, so a configured origin cannot redirect\nthe publisher to a private or plaintext destination.\n\n### Provenance hash rule\n\n`KAGGLE_NOTEBOOK_SHA256` is the SHA-256 of canonical immutable **cell sources**:\nthe header, preflight, CLI invocation, and artifact-display cells, in notebook\norder. The user configuration and the provenance-computing setup cell are excluded\nto avoid a self-referential hash. Each source is UTF-8, normalized to LF with one\ntrailing LF, and joined using one LF. The setup cell computes the value and the\nCLI records it only when publishing is enabled.\n", '# Validate the fixed public runtime before compiling or solving.\nimport subprocess\n\ngpu_names = [line.strip() for line in subprocess.check_output(\n    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True\n).splitlines() if line.strip()]\nif len(gpu_names) != 2 or any(name not in {"Tesla T4", "NVIDIA T4"} for name in gpu_names):\n    raise RuntimeError(f"this launcher requires exactly two T4 GPUs; observed={gpu_names!r}")\nprint("preflight_ok", {"gpu_names": gpu_names, "requested_beam": BEAM_WIDTH,\n                       "puzzle_ids": [PUZZLE_ID_START, PUZZLE_ID_END]})\n', '# The repository CLI owns model detection/export, profile selection, build, solve and validation.\nrun_config = {\n    "author_name": AUTHOR_NAME,\n    "checkpoint_path": str(CHECKPOINT_PATH),\n    "puzzle_info_json": str(PUZZLE_INFO_JSON),\n    "test_csv": str(TEST_CSV),\n    "sample_submission_csv": str(SAMPLE_SUBMISSION_CSV),\n    "puzzle_id_start": PUZZLE_ID_START,\n    "puzzle_id_end": PUZZLE_ID_END,\n    "beam_width": BEAM_WIDTH,\n    "max_depth": MAX_DEPTH,\n    "reflect_mode": REFLECT_MODE,\n    "reflect_source_csv": None if REFLECT_SOURCE_CSV is None else str(REFLECT_SOURCE_CSV),\n    "solution_mode": SOLUTION_MODE,\n    "collect_until_depth": COLLECT_UNTIL_DEPTH,\n    "max_collected_solutions": MAX_COLLECTED_SOLUTIONS,\n    "touch_bfs_radius": TOUCH_BFS_RADIUS,\n    "publish_results": PUBLISH_RESULTS,\n    "results_ingest_url": RESULTS_INGEST_URL,\n    "competition": COMPETITION,\n    "kaggle_owner": KAGGLE_OWNER,\n    "kaggle_slug": KAGGLE_SLUG,\n    "kaggle_version": KAGGLE_VERSION,\n    "kaggle_username": KAGGLE_USERNAME,\n    "solver_commit": SOLVER_COMMIT,\n    "kaggle_notebook_sha256": KAGGLE_NOTEBOOK_SHA256,\n}\nCONFIG_PATH.write_text(json.dumps(run_config, sort_keys=True) + "\\n", encoding="utf-8")\nrun_process = subprocess.run(\n    ["python", str(REPO / "tools" / "run_cayleypy_public.py"),\n     "--config-json", str(CONFIG_PATH), "--output-dir", str(OUTPUT_DIR)],\n    cwd=REPO,\n    check=False,\n)\nRUN_RETURN_CODE = run_process.returncode\nprint({"cli_return_code": RUN_RETURN_CODE})\n', '# Bounded handoff: show machine-readable statuses and a small solution preview.\nimport json\nimport pandas as pd\n\nfor name in ("selected_profile.json", "preflight.json", "publish_status.json", "run_summary.json"):\n    path = OUTPUT_DIR / name\n    print(f"\\n== {name} ==")\n    print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:6000] if path.is_file() else "missing")\n\nfor name in ("beam_run_results.csv", "solutions/solutions.csv", "submission.csv"):\n    path = OUTPUT_DIR / name\n    print(f"\\n== {name} ==")\n    if path.is_file():\n        display(pd.read_csv(path).head(20))\n    else:\n        print("missing")\n\nif RUN_RETURN_CODE != 0:\n    raise RuntimeError(f"public CLI failed with return code {RUN_RETURN_CODE}; artifacts above were retained")\n']
def canonical_cell_source(source):
    return source.replace("\r\n", "\n").replace("\r", "\n").rstrip("\n") + "\n"
KAGGLE_NOTEBOOK_SHA256 = sha256(
    "\n".join(canonical_cell_source(source) for source in NOTEBOOK_CELL_SOURCES).encode("utf-8")
).hexdigest()
print({"solver_commit": checked_commit, "notebook_source_sha256": KAGGLE_NOTEBOOK_SHA256})

In [ ]:
# Validate the fixed public runtime before compiling or solving.
import subprocess

gpu_names = [line.strip() for line in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).splitlines() if line.strip()]
if len(gpu_names) != 2 or any(name not in {"Tesla T4", "NVIDIA T4"} for name in gpu_names):
    raise RuntimeError(f"this launcher requires exactly two T4 GPUs; observed={gpu_names!r}")
print("preflight_ok", {"gpu_names": gpu_names, "requested_beam": BEAM_WIDTH,
                       "puzzle_ids": [PUZZLE_ID_START, PUZZLE_ID_END]})

In [ ]:
# The repository CLI owns model detection/export, profile selection, build, solve and validation.
run_config = {
    "author_name": AUTHOR_NAME,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "puzzle_info_json": str(PUZZLE_INFO_JSON),
    "test_csv": str(TEST_CSV),
    "sample_submission_csv": str(SAMPLE_SUBMISSION_CSV),
    "puzzle_id_start": PUZZLE_ID_START,
    "puzzle_id_end": PUZZLE_ID_END,
    "beam_width": BEAM_WIDTH,
    "max_depth": MAX_DEPTH,
    "reflect_mode": REFLECT_MODE,
    "reflect_source_csv": None if REFLECT_SOURCE_CSV is None else str(REFLECT_SOURCE_CSV),
    "solution_mode": SOLUTION_MODE,
    "collect_until_depth": COLLECT_UNTIL_DEPTH,
    "max_collected_solutions": MAX_COLLECTED_SOLUTIONS,
    "touch_bfs_radius": TOUCH_BFS_RADIUS,
    "publish_results": PUBLISH_RESULTS,
    "results_ingest_url": RESULTS_INGEST_URL,
    "competition": COMPETITION,
    "kaggle_owner": KAGGLE_OWNER,
    "kaggle_slug": KAGGLE_SLUG,
    "kaggle_version": KAGGLE_VERSION,
    "kaggle_username": KAGGLE_USERNAME,
    "solver_commit": SOLVER_COMMIT,
    "kaggle_notebook_sha256": KAGGLE_NOTEBOOK_SHA256,
}
CONFIG_PATH.write_text(json.dumps(run_config, sort_keys=True) + "\n", encoding="utf-8")
run_process = subprocess.run(
    ["python", str(REPO / "tools" / "run_cayleypy_public.py"),
     "--config-json", str(CONFIG_PATH), "--output-dir", str(OUTPUT_DIR)],
    cwd=REPO,
    check=False,
)
RUN_RETURN_CODE = run_process.returncode
print({"cli_return_code": RUN_RETURN_CODE})

In [ ]:
# Bounded handoff: show machine-readable statuses and a small solution preview.
import json
import pandas as pd

for name in ("selected_profile.json", "preflight.json", "publish_status.json", "run_summary.json"):
    path = OUTPUT_DIR / name
    print(f"\n== {name} ==")
    print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:6000] if path.is_file() else "missing")

for name in ("beam_run_results.csv", "solutions/solutions.csv", "submission.csv"):
    path = OUTPUT_DIR / name
    print(f"\n== {name} ==")
    if path.is_file():
        display(pd.read_csv(path).head(20))
    else:
        print("missing")

if RUN_RETURN_CODE != 0:
    raise RuntimeError(f"public CLI failed with return code {RUN_RETURN_CODE}; artifacts above were retained")